In [ ]:
# Industrial CNC Maintenance AI Assistant

This project uses Retrieval-Augmented Generation (RAG) to answer CNC maintenance questions based on the Haas CNC Operator's Manual.

Pipeline:
PDF → Text Splitting → Embeddings → FAISS → Mistral → Output Parser → Streamlit → ngrok

In [1]:
!pip install -q \
    faiss-cpu \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    langchain-text-splitters \
    langchain-classic \
    pypdf \
    sentence-transformers \
    "transformers==4.52.4" \
    bitsandbytes \
    accelerate \
    streamlit \
    pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 65.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.8/572.8 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 34.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 49.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 47.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 43.1 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into acc

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import torch

/tmp/ipykernel_49/2638423599.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
pdf_path = "/kaggle/input/datasets/youssefbadawy112/haas-pdf/haas_manual.pdf.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Number of pages:", len(documents))

Number of pages: 588


In [5]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 587


In [6]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = FAISS.from_documents(
    chunks,
    embedding
)

print("FAISS vector database created successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS vector database created successfully.


In [7]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_cfg,
    device_map="auto"
)

print("Mistral loaded successfully.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral loaded successfully.


In [9]:
from langchain_core.language_models.llms import LLM
from typing import Any

class CustomHFLLM(LLM):

    def _call(self, prompt: str, stop: Any = None) -> str:

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False
            )

        answer = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        return answer.strip()

    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"


llm = CustomHFLLM()

print("Custom LLM created successfully.")

Custom LLM created successfully.


In [10]:
from langchain_classic.output_parsers import (
    StructuredOutputParser,
    ResponseSchema
)

fault_schema = ResponseSchema(
    name="fault",
    description="The identified CNC machine fault or maintenance issue."
)

solution_schema = ResponseSchema(
    name="solution",
    description="The recommended maintenance action or solution for the CNC machine issue."
)

response_schemas = [
    fault_schema,
    solution_schema
]

output_parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)

format_instructions = output_parser.get_format_instructions()

print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"fault": string  // The identified CNC machine fault or maintenance issue.
	"solution": string  // The recommended maintenance action or solution for the CNC machine issue.
}
```


In [12]:
from langchain_core.prompts import PromptTemplate

maintenance_parser_template = """
You are an industrial CNC maintenance assistant.

Based ONLY on the provided context, identify the CNC maintenance issue
and give the recommended solution.

Context:
{context}

Question:
{question}

Respond ONLY in the following format:
{format_instructions}
"""

parser_prompt = PromptTemplate(
    template=maintenance_parser_template,
    input_variables=[
        "context",
        "question",
        "format_instructions"
    ]
)

In [14]:
def final_rag(query):

    # 1. Retrieve relevant documents
    docs = vectordb.similarity_search(query, k=3)

    # 2. Build context
    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    # 3. Build prompt
    formatted_prompt = parser_prompt.format(
        context=context,
        question=query,
        format_instructions=format_instructions
    )

    # 4. Generate response
    response = llm.invoke(formatted_prompt)

    # 5. Parse JSON output
    output_data = output_parser.parse(response)

    return output_data

In [15]:
question = "What should I do if the spindle has been idle for more than 4 days?"

result = final_rag(question)

print(result)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{'fault': 'Idle spindle for more than 4 days', 'solution': 'Run the spindle warm-up program (O09220) before using the machine.'}


In [42]:
%%writefile rag.py

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from langchain_core.language_models.llms import LLM
from langchain_core.prompts import PromptTemplate

from langchain_classic.output_parsers import (
    StructuredOutputParser,
    ResponseSchema
)

from typing import Any
import torch


# Load PDF
pdf_path = "/kaggle/input/datasets/youssefbadawy112/haas-pdf/haas_manual.pdf.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()


# Split text
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)


# Embeddings + FAISS
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = FAISS.from_documents(
    chunks,
    embedding
)


# Load Mistral
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_cfg,
    device_map="auto"
)


# Custom LLM
class CustomHFLLM(LLM):

    def _call(self, prompt: str, stop: Any = None) -> str:

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False
            )

        answer = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        return answer.strip()

    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"


llm = CustomHFLLM()


# Output Parser
fault_schema = ResponseSchema(
    name="fault",
    description="The identified CNC machine fault or maintenance issue."
)

solution_schema = ResponseSchema(
    name="solution",
    description="The recommended maintenance action or solution for the CNC machine issue."
)

response_schemas = [
    fault_schema,
    solution_schema
]

output_parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)

format_instructions = output_parser.get_format_instructions()


# Prompt
maintenance_parser_template = """
You are an industrial CNC maintenance assistant.

Based ONLY on the provided context, identify the CNC maintenance issue
and give the recommended solution.

Context:
{context}

Question:
{question}

Respond ONLY in the following format:
{format_instructions}
"""

parser_prompt = PromptTemplate(
    template=maintenance_parser_template,
    input_variables=[
        "context",
        "question",
        "format_instructions"
    ]
)


# Final RAG
def final_rag(query):

    docs = vectordb.similarity_search(query, k=3)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    formatted_prompt = parser_prompt.format(
        context=context,
        question=query,
        format_instructions=format_instructions
    )

    response = llm.invoke(formatted_prompt)

    output_data = output_parser.parse(response)

    return output_data

Overwriting rag.py


In [43]:
%%writefile app.py

import streamlit as st
from rag import final_rag

st.set_page_config(
    page_title="CNC Maintenance AI",
    page_icon="⚙️"
)

st.title("⚙️ CNC Maintenance AI Assistant")

st.write(
    "Ask a question about the Haas CNC machine."
)

question = st.text_input(
    "Maintenance Question:"
)

if st.button("Ask"):

    if question.strip():

        with st.spinner("Thinking..."):

            result = final_rag(question)

        st.subheader("Fault")
        st.write(result["fault"])

        st.subheader("Solution")
        st.write(result["solution"])

    else:

        st.warning("Please enter a question.")

Overwriting app.py


In [44]:
from pyngrok import ngrok
import subprocess

ngrok.kill()

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0"
    ]
)

print("Streamlit started successfully")

Streamlit started successfully




2026-10-08 03:47:16.750 Port 8501 is not available


In [45]:
from pyngrok import ngrok

public_url = ngrok.connect("127.0.0.1:8501")

print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://disposal-stiffen-reprogram.ngrok-free.dev" -> "http://127.0.0.1:8501"


In [46]:
try:
    process.terminate()
except:
    pass

from pyngrok import ngrok
import subprocess

ngrok.kill()

process = subprocess.Popen(
    [
        "streamlit",
        "run",
        "app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0"
    ]
)

print("Streamlit restarted successfully")

Streamlit restarted successfully




2026-10-08 03:47:26.602 Port 8501 is not available


In [48]:
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")

ngrok.set_auth_token(ngrok_token)

print("ngrok authentication configured successfully.")

ngrok authentication configured successfully.


In [49]:
from pyngrok import ngrok

public_url = ngrok.connect("127.0.0.1:8501")

print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://disposal-stiffen-reprogram.ngrok-free.dev" -> "http://127.0.0.1:8501"


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
